In [1]:
from pathlib import Path
import sys

import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'shared').exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'shared').exists() and (candidate / 'artifacts').exists():
            PROJECT_ROOT = candidate
            break

SRC_DIR = PROJECT_ROOT / 'artifacts' / 'ps-004-headcount-forecasting' / 'src'
MODELS_SRC_DIR = SRC_DIR / 'models'
for path in (SRC_DIR, MODELS_SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from feature_engineering import RAW_INPUT_PATH, OUTPUT_PATH, aggregate_profession_totals, build_features
from linear_model import DATE_STAMP, MODELS_DIR, train_linear_models

In [2]:
raw_df = pl.read_parquet(RAW_INPUT_PATH)
all_sector_df = raw_df.filter(pl.col('sector') == 'All')
print({'raw_shape': raw_df.shape, 'all_sector_shape': all_sector_df.shape})
profession_totals = aggregate_profession_totals(raw_df)
print({'profession_totals_shape': profession_totals.shape})
profession_totals

{'raw_shape': (264, 4), 'all_sector_shape': (0, 4)}
No sector='All' rows found in workforce_clean.parquet; falling back to profession-year sums across sectors.
{'profession_totals_shape': (48, 3)}


profession,year,count
cat,i32,i32
"""doctors""",2006,6931
"""doctors""",2007,7464
"""doctors""",2008,7841
"""doctors""",2009,8323
"""doctors""",2010,9030
…,…,…
"""physiotherapists""",2015,1549
"""physiotherapists""",2016,1693
"""physiotherapists""",2017,1814


In [3]:
features = build_features(profession_totals)
features.head(10)

profession,year,count,year_index,lag_1,lag_2
cat,i32,i32,i32,i32,i32
"""doctors""",2006,6931,0,null,null
"""doctors""",2007,7464,1,6931,null
"""doctors""",2008,7841,2,7464,6931
"""doctors""",2009,8323,3,7841,7464
"""doctors""",2010,9030,4,8323,7841
"""doctors""",2011,9646,5,9030,8323
"""doctors""",2012,10225,6,9646,9030
"""doctors""",2013,10953,7,10225,9646
"""doctors""",2014,11733,8,10953,10225


In [4]:
models = train_linear_models(features)
for profession, model in models.items():
    print(profession, {'coefficient': float(model.coef_[0]), 'intercept': float(model.intercept_)})

Training doctors: rows=11, years=2006-2016
Training nurses: rows=11, years=2006-2016
Training pharmacists: rows=11, years=2006-2016
Training physiotherapists: rows=3, years=2014-2016
doctors {'coefficient': 620.827272727273, 'intercept': 6675.136363636363}
nurses {'coefficient': 2080.2000000000003, 'intercept': 20791.272727272728}
pharmacists {'coefficient': 156.46363636363643, 'intercept': 1279.3181818181815}
physiotherapists {'coefficient': 149.49999999999994, 'intercept': 1395.8333333333333}


In [5]:
saved_features = pl.read_parquet(OUTPUT_PATH)
print({'features_shape': saved_features.shape})
saved_features.head(10)

{'features_shape': (48, 6)}


profession,year,count,year_index,lag_1,lag_2
cat,i32,i32,i32,i32,i32
"""doctors""",2006,6931,0,null,null
"""doctors""",2007,7464,1,6931,null
"""doctors""",2008,7841,2,7464,6931
"""doctors""",2009,8323,3,7841,7464
"""doctors""",2010,9030,4,8323,7841
"""doctors""",2011,9646,5,9030,8323
"""doctors""",2012,10225,6,9646,9030
"""doctors""",2013,10953,7,10225,9646
"""doctors""",2014,11733,8,10953,10225


In [6]:
model_files = sorted(path.name for path in MODELS_DIR.glob(f'*_linear_{DATE_STAMP}.pkl'))
model_files

['doctors_linear_20260423.pkl',
 'nurses_linear_20260423.pkl',
 'pharmacists_linear_20260423.pkl',
 'physiotherapists_linear_20260423.pkl']